# Kindle Review

#### Best Practices
- Preprocessing and cleaning
- Train test split
- BOW, TFIDF, Word2Vec
-  Train ML Model

In [82]:
import pandas as pd
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

In [83]:
# load the dataset
df = pd.read_csv('all_kindle_review.csv')

In [84]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [85]:
# we dont require all the columns. We only need reviewText and rating column
df = df[['reviewText', 'rating']]

In [86]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [87]:
df.shape

(12000, 2)

In [88]:
# Checking missing values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [89]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [90]:
df['rating'].value_counts() # so its a balanced dataset

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

#### Preprocessing and Cleaning

In [91]:
# for ratings below three we put it into negative review and three or above we put it into positive reviews in the rating column.
# NOTE: run this cell only once (Restart Kernel -> Run All). Re-running it turns every label into 0.

## negative review is 0 and positive review is 1
df['rating'] = df['rating'].apply(lambda x:0 if x<3 else 1)

In [92]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [93]:
df['rating'].unique() # so we converted 1,2 rating to 0 and 3,4,5 rating to 1

array([1, 0])

In [94]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [95]:
## Lower all the words in reviewText column

df['reviewText'] = df['reviewText'].str.lower()

In [96]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [97]:
# remove html tags
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

# remove url
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))(/[\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '', str(x)))

In [98]:
# removing special characters
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub('[^a-z A-Z 0-9]+','',x))

In [99]:
# remove stopwords, but keep negation words since they flip the sentiment (e.g. "not good")
stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor'}
df['reviewText'] = df['reviewText'].apply(lambda x:' '.join([y for y in x.split() if y not in stop_words]))

In [100]:
# remove any additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x:' '.join(x.split()))

In [101]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,not expect type book library pleased find pric...,1


#### Lemmatization

In [102]:
from nltk.stem import WordNetLemmatizer

In [103]:
lemmatizer = WordNetLemmatizer()

In [104]:
def lemmatize_words(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

In [105]:
df['reviewText'] = df['reviewText'].apply(lambda x: lemmatize_words(x))

In [106]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,not expect type book library pleased find pric...,1


#### Train test split

In [107]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.20, random_state=42, stratify=df['rating'])

#### Bag Of Words

In [108]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

#### TF-IDF

In [109]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [110]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 437400 stored elements and shape (9600, 36701)>

In [111]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 437400 stored elements and shape (9600, 36701)>

#### Naive Bayes (Multinomial)

In [112]:
from sklearn.naive_bayes import MultinomialNB

nb_model_bow = MultinomialNB().fit(X_train_bow, y_train)
nb_model_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)

In [113]:
nb_model_bow

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3200.,6400.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-1.1 ,-0.41]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 36701)","[[1.,5.,1.,...,0.,2.,0.], [1.,5.,0.,...,1.,5.,1.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 36701)","[[-11.55,-10.45,-11.55,...,-12.24,-11.14,-12.24], [-12.19,-11.09,-12.89,...,-12.19,-11.09,-12.19]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,36701


In [114]:
nb_model_tfidf

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3200.,6400.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-1.1 ,-0.41]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 36701)","[[0.51,0.52,0.03,...,0. ,0.39,0. ], [0.15,0.87,0. ,...,0.13,0.78,0.19]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 36701)","[[-10.49,-10.48,-10.87,...,-10.9 ,-10.57,-10.9 ], [-11.04,-10.55,-11.18,...,-11.05,-10.6 ,-11. ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,36701


In [115]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [116]:
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [117]:
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [118]:
confusion_matrix(y_test, y_pred_bow)

array([[ 585,  215],
       [ 149, 1451]])

In [122]:
print('BOW Accuracy Score :', accuracy_score(y_test, y_pred_bow))

BOW Accuracy Score : 0.8483333333333334


In [120]:
confusion_matrix(y_test, y_pred_tfidf)

array([[  73,  727],
       [   1, 1599]])

In [123]:
print('TFIDF Accuracy Score :', accuracy_score(y_test, y_pred_tfidf))

TFIDF Accuracy Score : 0.6966666666666667
